### IMPORTS

In [1]:
import warnings
# Ignore all FutureWarnings
warnings.filterwarnings("ignore", category=FutureWarning)

import re
import math
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn import tree
from sklearn.model_selection import RandomizedSearchCV, train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.svm import SVC
import random
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

# Make results reproducible
random.seed(100)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\UFC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


### Load Data

In [2]:
# Labelled data loading
data = pd.read_csv('A2_customer_churn_labeled.csv')
data.head()

,ID,credit_score,tenure,balance,number_of_products,has_credit_card,is_active_member,salary,customer_profile,Y
0,0,585,4,0.00,2,0,1,101728.46,This customer is a 44-year-old female from Spa...,0
1,1,743,6,140348.56,2,1,1,163254.39,This customer is a 32-year-old female from Ger...,0
2,2,527,10,136733.23,1,1,1,57589.29,This customer is a 41-year-old female from Ger...,0
3,3,732,6,98792.40,1,1,0,81491.70,This customer is a 45-year-old female from Ger...,1
4,4,641,3,0.00,2,1,0,116466.19,This customer is a 38-year-old female from Fra...,0


In [18]:
print('Shape of labeled data: ', data.shape)

Shape of labeled data:  (7000, 17)


### Helper functions for adding new columns 

In [3]:
def get_last_two_sentences(text):
    sentences = sent_tokenize(text)

    # Get the last 2 sentences
    last_two_sentences = sentences[-2:]

    return ' '.join(last_two_sentences)

# Create an object instance sih of SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

# Function that returns compound polarity score of the text
def get_polarity(text):
    # Get the polarity scores of the passed text
    return sia.polarity_scores(text)['compound']

# Load spaCy's English tokenizer and tagger
nlp = spacy.load("en_core_web_sm")

# Define a function to perform tokenization, stopwords removal, and lemmatization
def preprocess_text(text):
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if token.text.lower() not in STOP_WORDS]
    return " ".join(tokens)


### Add new columns

In [4]:
def add_new_columns(dataset):
     #  Just comment the line for the column that is not needed
    
    dataset['age'] = dataset['customer_profile'].apply(lambda x: int(re.findall('(\d+)-year-old', x)[0]))
    dataset['gender'] = dataset['customer_profile'].apply(lambda x: re.findall('(male|female)', x)[0])
    dataset['country'] = dataset['customer_profile'].apply(lambda x: re.findall('from (\w+)', x)[0])
    dataset['last_lines'] = dataset['customer_profile'].apply(lambda text: get_last_two_sentences(text))
    dataset['customer_profile_polarity'] = dataset['customer_profile'].apply(lambda text: get_polarity(text))
    dataset['last_lines_polarity'] = dataset['last_lines'].apply(lambda text: get_polarity(text))
    dataset['customer_profile_tokenized'] = dataset['customer_profile'].apply(lambda text: preprocess_text(text))
    return dataset

In [5]:
data1 = add_new_columns(data)

### Drop any existing columns

In [6]:
def drop_any_existing_columns(dataset, columns = ['ID']):
    dataset = dataset.drop(columns=columns, inplace=False)
    return dataset

In [7]:
columns_to_drop = ['ID', 'customer_profile', 'last_lines', 'customer_profile_tokenized']

data2 = drop_any_existing_columns(data1, columns = columns_to_drop)

### Any required feature transformations

In [8]:
data2.head()

,credit_score,tenure,balance,number_of_products,has_credit_card,is_active_member,salary,Y,age,gender,country,customer_profile_polarity,last_lines_polarity
0,585,4,0.00,2,0,1,101728.46,0,44,female,Spain,0.7311,0.4512
1,743,6,140348.56,2,1,1,163254.39,0,32,female,Germany,0.7845,0.6486
2,527,10,136733.23,1,1,1,57589.29,0,41,female,Germany,0.7845,0.6486
3,732,6,98792.40,1,1,0,81491.70,1,45,female,Germany,0.4482,-0.3089
4,641,3,0.00,2,1,0,116466.19,0,38,female,France,0.4482,-0.3089


In [9]:
### Dummy variablize
data3 = pd.get_dummies(data2, columns = ['gender', 'country'], drop_first=True )
print(data3.columns)
data3.head()

Index(['credit_score', 'tenure', 'balance', 'number_of_products',
       'has_credit_card', 'is_active_member', 'salary', 'Y', 'age',
       'customer_profile_polarity', 'last_lines_polarity', 'gender_male',
       'country_Germany', 'country_Spain'],
      dtype='object')


,credit_score,tenure,balance,number_of_products,has_credit_card,is_active_member,salary,Y,age,customer_profile_polarity,last_lines_polarity,gender_male,country_Germany,country_Spain
0,585,4,0.00,2,0,1,101728.46,0,44,0.7311,0.4512,False,False,True
1,743,6,140348.56,2,1,1,163254.39,0,32,0.7845,0.6486,False,True,False
2,527,10,136733.23,1,1,1,57589.29,0,41,0.7845,0.6486,False,True,False
3,732,6,98792.40,1,1,0,81491.70,1,45,0.4482,-0.3089,False,True,False
4,641,3,0.00,2,1,0,116466.19,0,38,0.4482,-0.3089,False,False,False


## Models implementation

### Model evaluation metrics

In [14]:
def calc_f1_score(model, X=None, y=None, dataset=None, type='macro'):
    if X is None and y is None and dataset is not None:
        y = dataset['Y']
        X = dataset.drop(columns='Y')
    y_pred = model.predict(X)
    return f1_score(y, y_pred, average='macro')

### Model 1 SVM Basic

In [11]:
# Perform train test split
X_train, X_test, y_train, y_test = train_test_split(data3.drop(columns='Y'), data3['Y'], test_size=0.2, random_state=42)

In [12]:
svm_classifier = SVC(kernel='linear', C=1)

In [13]:
svm_classifier.fit(X_train, y_train)

SVC(C=1, kernel='linear')

In [15]:
calc_f1_score(svm_classifier, X=X_test, y=y_test)

0.4792467428123652

In [16]:
svm_classifier.fit(data3.drop(columns='Y'), data3['Y'])

SVC(C=1, kernel='linear')

In [ ]:
### Model 2

### Generating the Kaggle Submission File

In [20]:
final_model = svm_classifier

In [29]:
X_kaggle_test = pd.read_csv('A2_customer_churn_kaggle.csv')

In [30]:
# Add columns
X_kaggle_test1 = add_new_columns(X_kaggle_test)

In [31]:
# Drop columns
columns_to_drop = ['ID', 'Y', 'customer_profile', 'last_lines', 'customer_profile_tokenized']
X_kaggle_test1 = drop_any_existing_columns(X_kaggle_test1, columns = columns_to_drop)

In [32]:
### Dummy variablize
X_kaggle_test1 = pd.get_dummies(X_kaggle_test1, columns = ['gender', 'country'], drop_first=True )

In [33]:
print('Shape of kaggle test: ', X_kaggle_test1.shape)
X_kaggle_test1.head()

Shape of kaggle test:  (2010, 13)


,credit_score,tenure,balance,number_of_products,has_credit_card,is_active_member,salary,age,customer_profile_polarity,last_lines_polarity,gender_male,country_Germany,country_Spain
0,652,4,59486.31,1,1,0,163944.19,48,0.4482,-0.3089,True,False,False
1,714,4,0.00,2,1,1,37605.90,29,0.7845,0.6486,True,False,True
2,733,3,100337.96,3,1,0,48559.19,34,0.4482,-0.3089,True,True,False
3,577,8,79757.21,1,1,0,135650.72,43,0.4482,-0.3089,True,False,True
4,600,2,119755.00,1,1,1,21852.91,30,0.7845,0.6486,True,True,False


In [34]:
y_pred = final_model.predict(X_kaggle_test1)
df_pred = pd.concat([X_kaggle_test['ID'], pd.DataFrame(y_pred,columns=['Y'])], axis = 1)
df_pred.to_csv('kaggle_pred_values.csv',index=False)